In [1]:
import threading
from concurrent.futures import ThreadPoolExecutor
import queue
from msg_type import DataReady, PipelineDone, Config, Token

In [2]:
class ActorSystem:
    """A lightweight actor scheduler."""
    
    def __init__(self, num_workers=2):
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.actors = {}                        
        self.task_queue = queue.Queue()          
        self.states = ['idle', 'in-queue', 'processing']
        self._running = True
        self._start_workers()


    # ---- actor registration ----

    def create_actor(self, actor):
        self.actors[actor.name] = {
            'actor': actor,
            'state': self.states[0]             
        }

    # ---- messaging ----

    def send_message(self, target_name, message):
        target = self.actors.get(target_name)
        if not target:
            print(f'unknown actor: {target_name}')
            return

        target['actor'].store_message(message)          # step 1: store first
        if target['state'] == 'idle':                    # step 2: then check
            self.task_queue.put(target['actor'])
            self._advance_state(target_name)             # idle -> in-queue

    # ---- scheduling internals ----

    def _start_workers(self):
        """Launch worker threads that pull actors from the task queue."""
        for _ in range(self.executor._max_workers):
            t = threading.Thread(target=self._worker_loop, daemon=True)
            t.start()

    def _worker_loop(self):
        """Main loop for each worker thread."""
        while self._running:
            try:
                actor = self.task_queue.get(timeout=0.1)
            except queue.Empty:
                continue
            if not self._running:
                break
            self._advance_state(actor.name)
            future = self.executor.submit(actor)
            future.add_done_callback(
                lambda _, name=actor.name: self._on_actor_done(name)
            )

    def _on_actor_done(self, actor_name):
        """Called when an actor finishes processing."""
        self._advance_state(actor_name)                  # processing -> idle
        if not self.actors[actor_name]['actor'].mailbox.empty():
            self.task_queue.put(self.actors[actor_name]['actor'])
            self._advance_state(actor_name)              # idle -> in-queue

    def _advance_state(self, actor_name):
        """Cycle actor state: idle -> in-queue -> processing -> idle."""
        current = self.actors[actor_name]['state']
        next_idx = (self.states.index(current) + 1) % len(self.states)
        self.actors[actor_name]['state'] = self.states[next_idx]

    def shutdown(self):
        self._running = False
        self.executor.shutdown(wait=False) 


In [3]:
class Actor:
    """Base actor with mailbox and behavior switching (become pattern)."""
 
    def __init__(self, name, system):
        self.name = name
        self.system = system
        self.mailbox = queue.Queue()
        self.behavior = self._default_behavior

    def store_message(self, message):
        self.mailbox.put(message)
 
    def __call__(self):
        while not self.mailbox.empty():
            message = self.mailbox.get()
            self.behavior(message)
 
    def become(self, new_behavior):
        self.behavior = new_behavior
 
    def _default_behavior(self, message):
        raise NotImplementedError(
            f'{self.name}: no behavior set for message {message}')   

In [4]:
class PipelinedActor(Actor):
    """Base class for actors that dispatch work to a FlowGraph pipeline.

    Handles: slot pool, queue, backpressure, dispatch, completion.
    Subclass implements: _fill_slot(), on_result(), on_config().
    """

    def __init__(self, name, system, pool, graph, first_node):
        super().__init__(name, system)
        self._pool = pool
        self._graph = graph
        self._queue = []
        self._in_flight = 0
        self._first_node = first_node

    def _default_behavior(self, msg):
        if isinstance(msg, DataReady):
            self._queue.append(msg)
            self._try_dispatch()

        elif isinstance(msg, PipelineDone):
            self.on_result(msg)           # subclass reads slot BEFORE release
            self._pool.release(msg.slot)
            self._in_flight -= 1
            self._try_dispatch()

        elif isinstance(msg, Config):
            self.on_config(msg)

    def _try_dispatch(self):
        while self._queue:
            slot_idx = self._pool.acquire()
            if slot_idx is None:
                break                     # all slots busy, wait for pipeline_done
            msg = self._queue.pop(0)
            self._fill_slot(slot_idx, msg)
            self._in_flight += 1
            token = Token(slot=slot_idx, tag=msg.tag)
            self._graph.try_put(self._first_node, token)

    def _fill_slot(self, slot_idx, msg):
        """Read data from source, write into pool slot."""
        raise NotImplementedError

    def on_result(self, msg):
        """Process pipeline result. Slot is still valid here."""
        pass

    def on_config(self, msg):
        """Handle config message."""
        pass

#### Test
1. actor registration

In [5]:
import time

In [6]:
# Helper function: wait until all actors are idle 
def wait_until_idle(system, timeout=2.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if all(info['state'] == 'idle' for info in system.actors.values()):
            time.sleep(0.05)  # small extra margin for callbacks
            return True
        time.sleep(0.01)
    return False

In [7]:
# test actor system
system = ActorSystem(num_workers=2)

class DummyActor(Actor):
    def handle(self, message):
        pass
 
a = DummyActor('alice', system)
system.create_actor(a)
 
assert 'alice' in system.actors
assert system.actors['alice']['state'] == 'idle'
assert system.actors['alice']['actor'] is a

2. single message delivery

In [8]:
system = ActorSystem(num_workers=2)

class CollectorActor(Actor):
    def __init__(self, name, system):
        super().__init__(name, system)
        self.received = []

    def _default_behavior(self, message):
        self.received.append(message)

bob = CollectorActor('bob', system)
system.create_actor(bob)

system.send_message('bob', 'hello')
wait_until_idle(system)

assert bob.received == ['hello']
assert system.actors['bob']['state'] == 'idle'

3. multiple messages batched

In [9]:
system = ActorSystem(num_workers=2)
carol = CollectorActor('carol', system)
system.create_actor(carol)
 
# send 5 messages rapidly — actor should batch them in one activation
for i in range(5):
    system.send_message('carol', f'msg_{i}')
 
wait_until_idle(system)
 
assert len(carol.received) == 5
for i in range(5):
    assert f'msg_{i}' in carol.received

4. actor-to-actor forwarding

In [10]:
system = ActorSystem(num_workers=2)
 
class ForwarderActor(Actor):
    def __init__(self, name, system, forward_to):
        super().__init__(name, system)
        self.forward_to = forward_to

    def _default_behavior(self, message):
        self.system.send_message(self.forward_to, f'fwd:{message}')
 
sink = CollectorActor('sink', system)
fwd = ForwarderActor('fwd', system, 'sink')
system.create_actor(sink)
system.create_actor(fwd)
 
system.send_message('fwd', 'data')
wait_until_idle(system)
 
assert sink.received == ['fwd:data']
assert system.actors['fwd']['state'] == 'idle'
assert system.actors['sink']['state'] == 'idle'

5. unknown actor

In [11]:
system = ActorSystem(num_workers=2)
dummy = CollectorActor('dummy', system)
system.create_actor(dummy)
 
# should print warning but not crash
system.send_message('nonexistent', 'hello')
wait_until_idle(system)
 
assert dummy.received == []  # dummy is unaffected

unknown actor: nonexistent


6. message passing: 10 actors x 20 messages each

In [12]:
system = ActorSystem(num_workers=4)
actors = []
for i in range(10):
    a = CollectorActor(f'actor_{i}', system)
    system.create_actor(a)
    actors.append(a)
 
for i in range(10):
    for j in range(20):
        system.send_message(f'actor_{i}', f'msg_{j}')
 
wait_until_idle(system)
 
for a in actors:
    assert len(a.received) == 20

7. pipeline actor: single message through pipeline

In [ ]:
class TestSlot:
    def __init__(self):
        self.data = 0
        self.result = 0
    def reset(self):
        self.data = 0
        self.result = 0

def double(slot):
    """Pure function: doubles slot.data into slot.result."""
    slot.result = slot.data * 2

def add_ten(slot):
    """Pure function: adds 10 to slot.result."""
    slot.result = slot.result + 10

# ---- test actor ----

class TestPipelinedActor(PipelinedActor):
    """Simple subclass: fills slot with tag value, sends result to controller."""

    def __init__(self, system, pool, graph, first_node):
        super().__init__('worker', system, pool, graph, first_node)

    def _fill_slot(self, slot_idx, msg):
        slot = self._pool.slots[slot_idx]
        slot.data = msg.tag['value']

    def on_result(self, msg):
        slot = self._pool.slots[msg.slot]
        self.system.send_message('controller', {
            'result': slot.result,
            'tag': msg.tag
        })

In [14]:
from threading import Event
from flow_graph import FlowGraph, FunctionNode, SlotPool

In [15]:
system = ActorSystem(num_workers=2)
done = Event()

pool = SlotPool(3, lambda: TestSlot())

graph = FlowGraph(num_workers=2)
n1 = FunctionNode('double', pool.make_stage(double), concurrency=1)
n2 = FunctionNode('add_ten', pool.make_stage(add_ten), concurrency=1,
                  done_callback=lambda token: system.send_message(
                      'worker', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(n1, n2)

worker = TestPipelinedActor(system, pool, graph, n1)
system.create_actor(worker)

collector = CollectorActor('controller', system)
system.create_actor(collector)

# send: data=5 → double → 10 → add_ten → 20
system.send_message('worker', DataReady(pid=0, tag={'value': 5}))

wait_until_idle(system)
graph.shutdown()

assert len(collector.received) == 1
assert collector.received[0]['result'] == 20   # 5 * 2 + 10
assert collector.received[0]['tag']['value'] == 5

8. pipeline actor: backpressure (more messages than slots)

In [17]:
system = ActorSystem(num_workers=2)
done = Event()

pool = SlotPool(2, lambda: TestSlot())   # only 2 slots

graph = FlowGraph(num_workers=2)
n1 = FunctionNode('double', pool.make_stage(double), concurrency=1)
n2 = FunctionNode('add_ten', pool.make_stage(add_ten), concurrency=1,
                  done_callback=lambda token: system.send_message(
                      'worker', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(n1, n2)

worker = TestPipelinedActor(system, pool, graph, n1)
system.create_actor(worker)

collector = CollectorActor('controller', system)
system.create_actor(collector)

# send 5 messages with only 2 slots — backpressure must queue the rest
for i in range(5):
    system.send_message('worker', DataReady(pid=0, tag={'value': i}))

wait_until_idle(system)
graph.shutdown()

results = sorted([r['result'] for r in collector.received])
expected = sorted([i * 2 + 10 for i in range(5)])
assert results == expected
assert len(collector.received) == 5